In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split, KFold
import re
import os
import numpy as np
from tqdm import tqdm
import time

class NovelDataset(Dataset):
    """
    小说数据集类，继承自torch.utils.data.Dataset
    
    功能:
        1. 处理文本数据
        2. 将文本转换为模型可用的输入格式
        3. 提供数据集的长度和索引访问功能
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        """
        初始化数据集
        
        参数:
            texts (list): 文本内容列表
            labels (list): 标签列表
            tokenizer: BERT分词器
            max_length (int): 最大文本长度
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        """返回数据集长度"""
        return len(self.texts)

    def __getitem__(self, idx):
        """
        获取指定索引的数据项
        
        参数:
            idx (int): 数据索引
        
        返回:
            dict: 包含输入ID、注意力掩码和标签的字典
        """
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def is_chapter_title(text):
    """
    判断文本是否为章节标题
    
    参数:
        text (str): 待判断的文本
    
    功能:
        1. 使用多种正则表达式模式匹配章节标题
        2. 识别不同类型的章节标题格式
    
    返回:
        tuple: (is_title, title_type)
            - is_title: 是否为章节标题
            - title_type: 标题类型（'volume', 'chapter', 'volume_chapter', 'other'）
    """
    patterns = [
        # 格式1：带括号的章节
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+（[一二三四五六七八九十\d]+）',
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^(]+\([一二三四五六七八九十\d]+\)',
        
        # 格式2：卷带书名号
        r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》',
        r'^第\d+卷\s*《[^》]+》',
        
        # 格式2和3：基本章回格式
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+章\s*[^（(]+',
        r'^第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^第\d+回\s*[^（(]+',
        
        # 格式4：回目格式
        r'^第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^第\d+回\s*[^（(]+',
        
        # 格式5：卷章连体
        r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章',
        r'^第\d+卷第\d+章',
        
        # 其他常见格式
        r'^楔子\s*[^（(]*',
        r'^序章\s*[^（(]*',
        r'^尾声\s*[^（(]*',
        r'^Chapter\s*\d+\s*[^(]*',
        r'^[一二三四五六七八九十\d]+[、.]\s*[^（(]*',
        
        # 带引号的格式
        r'^『[一二三四五六七八九十\d]+』\s*[^（(]*',
        r'^「[一二三四五六七八九十\d]+」\s*[^（(]*',
        r'^《[一二三四五六七八九十\d]+》\s*[^（(]*',
        
        # 带括号的格式
        r'^（[一二三四五六七八九十\d]+）\s*[^（(]*',
        r'^\([一二三四五六七八九十\d]+\)\s*[^（(]*',
        
        # 英文数字格式
        r'^[Oo]ne\b\s*[^(]*',
        r'^[Tt]wo\b\s*[^(]*',
        r'^[Tt]hree\b\s*[^(]*',
        r'^[Ff]our\b\s*[^(]*',
        r'^[Ff]ive\b\s*[^(]*',
        r'^[Ss]ix\b\s*[^(]*',
        r'^[Ss]even\b\s*[^(]*',
        r'^[Ee]ight\b\s*[^(]*',
        r'^[Nn]ine\b\s*[^(]*',
        r'^[Tt]en\b\s*[^(]*'
    ]
    
    # 检查是否是卷标题
    volume_pattern = r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》'
    if re.match(volume_pattern, text):
        return True, 'volume'
    
    # 检查是否是章标题
    chapter_pattern = r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+'
    if re.match(chapter_pattern, text):
        return True, 'chapter'
    
    # 检查是否是卷章连体格式
    volume_chapter_pattern = r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章'
    if re.match(volume_chapter_pattern, text):
        return True, 'volume_chapter'
    
    # 检查其他格式
    return any(re.match(pattern, text) for pattern in patterns), 'other'

def extract_chapter_info(text):
    """
    提取章节信息，返回层级和编号
    
    参数:
        text (str): 章节标题文本
    
    功能:
        1. 提取卷号
        2. 提取章号
        3. 处理卷章连体格式
    
    返回:
        dict: 包含卷号、章节号和原始文本的信息字典
    """
    # 提取卷号
    volume_match = re.match(r'^第([一二三四五六七八九十百千万\d]+)卷', text)
    volume_num = volume_match.group(1) if volume_match else None
    
    # 提取章号
    chapter_match = re.match(r'^第([一二三四五六七八九十百千万\d]+)章', text)
    chapter_num = chapter_match.group(1) if chapter_match else None
    
    # 提取卷章连体格式
    volume_chapter_match = re.match(r'^第([一二三四五六七八九十百千万\d]+)卷第([一二三四五六七八九十百千万\d]+)章', text)
    if volume_chapter_match:
        volume_num = volume_chapter_match.group(1)
        chapter_num = volume_chapter_match.group(2)
    
    return {
        'volume': volume_num,
        'chapter': chapter_num,
        'text': text
    }

def prepare_data(file_path):
    """
    准备数据，处理单个文件
    
    参数:
        file_path (str): 小说文件路径
    
    功能:
        1. 读取文件内容
        2. 识别章节标题和正文
        3. 提取章节信息
        4. 生成标签
    
    返回:
        tuple: (texts, labels, chapter_info)
            - texts: 文本内容列表
            - labels: 标签列表（1表示标题，0表示正文）
            - chapter_info: 章节信息列表
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    texts = []
    labels = []
    chapter_info = []
    current_volume = None
    
    # 标注章节与正文
    for line in lines:
        line = line.strip()
        if line:
            texts.append(line)
            is_title, title_type = is_chapter_title(line)
            labels.append(1 if is_title else 0)
            
            if is_title:
                info = extract_chapter_info(line)
                if info['volume']:
                    current_volume = info['volume']
                elif info['chapter']:
                    info['volume'] = current_volume
                chapter_info.append(info)
            else:
                chapter_info.append({
                    'volume': current_volume,
                    'chapter': None,
                    'text': line
                })
    
    return texts, labels, chapter_info

def train_model(model, train_loader, val_loader, device, num_epochs=10, patience=3):
    """
    训练模型
    
    参数:
        model: 要训练的模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        device: 训练设备（CPU或GPU）
        num_epochs (int): 训练轮数
        patience (int): 早停耐心值
    
    功能:
        1. 设置优化器和学习率调度器
        2. 进行模型训练
        3. 实现早停机制
        4. 保存最佳模型
    
    返回:
        训练好的模型
    """
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    # 添加学习率调度器
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        start_time = time.time()
        
        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}'):
            # 确保数据在正确的设备上
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        # 验证
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Validating'):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                total_val_loss += loss.item()
                
                predictions = torch.argmax(outputs.logits, dim=-1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        
        avg_val_loss = total_val_loss / len(val_loader)
        accuracy = correct / total
        
        # 早停检查
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    # 恢复最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model

def organize_chapters(texts, labels, chapter_info):
    """
    组织章节和正文的层级关系
    
    参数:
        texts (list): 文本内容列表
        labels (list): 标签列表，1表示标题，0表示正文
        chapter_info (list): 章节信息列表，包含卷号、章节号等信息
    
    功能:
        1. 将文本内容按照卷和章节的层级关系组织
        2. 记录每个章节的标题和内容
        3. 维护卷和章节的层级关系
    
    返回:
        list: 组织好的章节内容列表，每个元素包含卷号、章节号、章节标题和内容
    """
    organized_content = []
    current_volume = None
    current_chapter = None
    current_content = []
    current_chapter_title = None
    
    for i, (text, label, info) in enumerate(zip(texts, labels, chapter_info)):
        if label == 1:  # 如果是标题
            # 保存之前的内容
            if current_chapter is not None:
                organized_content.append({
                    'volume': current_volume,
                    'chapter': current_chapter,
                    'chapter_title': current_chapter_title,
                    'content': current_content
                })
                current_content = []
            
            # 处理新的标题
            if info['volume']:
                current_volume = info['volume']
                current_chapter = None
                current_chapter_title = text
            elif info['chapter']:
                current_chapter = info['chapter']
                current_chapter_title = text
        else:  # 如果是正文
            current_content.append(text)
    
    # 保存最后一个章节的内容
    if current_chapter is not None:
        organized_content.append({
            'volume': current_volume,
            'chapter': current_chapter,
            'chapter_title': current_chapter_title,
            'content': current_content
        })
    
    return organized_content

def save_chapters_to_files(organized_content, book_name):
    """
    将组织好的章节内容保存为单独的文件
    
    参数:
        organized_content (list): 组织好的章节内容列表，每个元素包含卷号、章节号、章节标题和内容
        book_name (str): 书籍名称，用于创建对应的文件夹
    
    功能:
        1. 创建以书籍名命名的文件夹
        2. 遍历所有章节，将每个章节保存为单独的文件
        3. 文件名格式：有卷的为"第X卷第X章.txt"，无卷的为"第X章.txt"
        4. 文件内容包含章节标题和正文内容
    """
    # 创建书籍文件夹
    book_dir = os.path.join('organized_books', book_name)
    os.makedirs(book_dir, exist_ok=True)
    
    for item in organized_content:
        # 构建文件名
        if item['volume']:
            filename = f"第{item['volume']}卷第{item['chapter']}章.txt"
        else:
            filename = f"第{item['chapter']}章.txt"
        
        # 构建文件路径
        file_path = os.path.join(book_dir, filename)
        
        # 写入文件内容
        with open(file_path, 'w', encoding='utf-8') as f:
            # 写入章节标题
            if item['chapter_title']:
                f.write(item['chapter_title'] + '\n\n')
            
            # 写入章节内容
            for line in item['content']:
                f.write(line + '\n')

def format_output(organized_content, book_name):
    """
    格式化输出内容并保存到文件
    
    参数:
        organized_content (list): 组织好的章节内容列表
        book_name (str): 书籍名称
    
    功能:
        1. 调用save_chapters_to_files保存每个章节为单独的文件
        2. 生成完整的书籍内容
        3. 返回格式化后的完整内容列表
    
    返回:
        list: 包含完整书籍内容的列表
    """
    # 保存章节到单独的文件
    save_chapters_to_files(organized_content, book_name)
    
    # 同时保存一个完整的文件
    output = []
    current_volume = None
    
    for item in organized_content:
        # 如果有卷且与当前卷不同，输出卷标题
        if item['volume'] and item['volume'] != current_volume:
            output.append(f"第{item['volume']}卷")
            current_volume = item['volume']
        
        # 输出章节标题
        if item['chapter']:
            output.append(f"第{item['chapter']}章")
        
        # 输出章节内容
        output.extend(item['content'])
    
    return output

def main():
    """
    主函数
    
    功能:
        1. 设置训练设备
        2. 加载预训练模型和分词器
        3. 准备训练数据
        4. 进行K折交叉验证
        5. 训练和评估模型
        6. 保存模型和分词器
    """
    # 检查CUDA是否可用
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'使用设备: {device}')
        print(f'GPU型号: {torch.cuda.get_device_name(0)}')
        print(f'GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')
        # 设置默认设备
        torch.cuda.set_device(0)
        # 清空GPU缓存
        torch.cuda.empty_cache()
        # 设置内存分配策略
        torch.cuda.set_per_process_memory_fraction(0.8)  # 限制GPU内存使用为80%
    else:
        device = torch.device('cpu')
        print('警告: CUDA不可用，将使用CPU训练')
    
    # 加载预训练模型和分词器
    model_name = 'bert-base-chinese'
    print('正在加载预训练模型...')
    tokenizer = BertTokenizer.from_pretrained(model_name)
    # 使用fp16精度加载模型
    model = BertForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=2,
        torch_dtype=torch.float16  # 使用半精度浮点数
    )
    model = model.to(device)
    print('模型加载完成')
    
    # 准备数据
    all_texts = []
    all_labels = []
    
    print('正在读取数据...')
    # 读取所有小说文件
    for file in os.listdir('books'):
        if file.endswith('.txt'):
            file_path = os.path.join('books', file)
            print(f'处理文件: {file}')
            texts, labels, _ = prepare_data(file_path)
            all_texts.extend(texts)
            all_labels.extend(labels)
    
    print(f'总数据量: {len(all_texts)}')
    
    # 使用K折交叉验证
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(all_texts)):
        print(f'\n训练第 {fold + 1}/3 折')
        print(f'训练集大小: {len(train_idx)}')
        print(f'验证集大小: {len(val_idx)}')
        
        train_texts = [all_texts[i] for i in train_idx]
        train_labels = [all_labels[i] for i in train_idx]
        val_texts = [all_texts[i] for i in val_idx]
        val_labels = [all_labels[i] for i in val_idx]
        
        # 创建数据集和数据加载器
        train_dataset = NovelDataset(train_texts, train_labels, tokenizer)
        val_dataset = NovelDataset(val_texts, val_labels, tokenizer)
        
        # 修改数据加载器配置
        train_loader = DataLoader(
            train_dataset, 
            batch_size=8,  # 进一步减小batch_size
            shuffle=True, 
            num_workers=0,
            pin_memory=True,
            drop_last=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=8,
            num_workers=0,
            pin_memory=True
        )
        
        # 训练模型
        model = train_model(model, train_loader, val_loader, device)
        
        # 评估当前折
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask)
                predictions = torch.argmax(outputs.logits, dim=-1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        
        fold_accuracy = correct / total
        fold_scores.append(fold_accuracy)
        print(f'第 {fold + 1} 折准确率: {fold_accuracy:.4f}')
    
    # 输出平均准确率
    mean_accuracy = np.mean(fold_scores)
    std_accuracy = np.std(fold_scores)
    print(f'\n交叉验证结果:')
    print(f'平均准确率: {mean_accuracy:.4f} ± {std_accuracy:.4f}')
    
    # 保存最终模型和分词器
    print('保存模型...')
    model.save_pretrained('./models')
    tokenizer.save_pretrained('./models')
    print('模型保存完成')

if __name__ == '__main__':
    main() 

使用设备: cuda
GPU型号: NVIDIA GeForce RTX 3060 Laptop GPU
GPU内存: 6.00 GB
正在加载预训练模型...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


模型加载完成
正在读取数据...
处理文件: 余罪.txt
处理文件: 大奉打更人.txt
处理文件: 射雕英雄传.txt
处理文件: 平步青云.txt
处理文件: 开端.txt
处理文件: 超级兵王.txt
总数据量: 274495

训练第 1/3 折
训练集大小: 182996
验证集大小: 91499


Validating: 100%|██████████| 11438/11438 [03:25<00:00, 55.77it/s]


第 1 折准确率: 0.9903

训练第 2/3 折
训练集大小: 182997
验证集大小: 91498


Validating: 100%|██████████| 11438/11438 [06:05<00:00, 31.31it/s]  


第 2 折准确率: 0.9901

训练第 3/3 折
训练集大小: 182997
验证集大小: 91498


Validating: 100%|██████████| 11438/11438 [03:21<00:00, 56.74it/s]


第 3 折准确率: 0.9906

交叉验证结果:
平均准确率: 0.9903 ± 0.0002
保存模型...
模型保存完成
